# v0.32.0 — Materialized views

## 1. Intro

Since v0.32.0 a model can read a **materialized view**: a table SurrealDB computes from a `SELECT` and keeps up to date
as its source table changes — counts, sums, filtered copies. The view is a read-only model:

- `SurrealConfigDict(view_query="SELECT …")` declares it on the model, checked when the class is defined;
- `view_ddl()` renders its `DEFINE TABLE … AS (…)` without a connection, `await define_view()` applies it idempotently
  (`OVERWRITE`) and `await remove_view()` removes it;
- it reads like any model — filters, `get()`, `watch()`, transactions, the query cache — and every ORM write of it
  raises `TypeError`: SurrealDB computes a view's records itself, so the ORM writes to its source tables instead.

The same code runs on SurrealDB 2.x and 3.x, except the rows of the README's 2.x/3.x table. The notebook runs top to
bottom on both lines: only the server line (§2), the live notifications (§7), the transaction (§8), the grouped
average (§9) and the removal of a view that another view reads (§12) differ.

## 2. Connect

`SURREALDB_HOST` and `SURREALDB_PORT` choose the server (default `localhost:8000`), over WebSocket. The views live in
their own database, `examples_views`, reached through the named connection `views_demo`. The notebook starts by
resetting that database, so it can be re-executed in place: `REMOVE DATABASE` and `DEFINE DATABASE` go in **one**
query, because on SurrealDB 3.x a bare `REMOVE DATABASE` leaves the session failing until the database is defined
again. The same query defines the `orders` table: a view's source table must exist before the view is defined.

`apply_ddl()` runs SurrealQL through the SDK client's `query_raw()` and checks the status of **every** statement
(`client.query()` checks only the first one's).

In [1]:
import asyncio
import contextlib
import os

from surreal_orm_lite import SurrealDBConnectionManager

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")
SERVER_URL = f"ws://{HOST}:{PORT}/rpc"

SurrealDBConnectionManager.set_connection(  # the default connection
    url=SERVER_URL,
    user="root",
    password="root",
    namespace="examples",
    database="examples",
)
SurrealDBConnectionManager.add_connection(  # the database this notebook defines its views in
    "views_demo",
    url=SERVER_URL,
    user="root",
    password="root",
    namespace="examples",
    database="examples_views",
)


async def apply_ddl(client, ddl: str) -> int:
    """Run SurrealQL statements and check every one of them; return how many ran."""
    response = await client.query_raw(ddl, {})
    if "error" in response:
        raise RuntimeError(response["error"])
    refused = [result["result"] for result in response["result"] if result["status"] != "OK"]
    if refused:
        raise RuntimeError(refused)
    return len(response["result"])


demo = await SurrealDBConnectionManager.get_client("views_demo")
await apply_ddl(
    demo,
    "REMOVE DATABASE IF EXISTS examples_views; DEFINE DATABASE examples_views; DEFINE TABLE orders TYPE ANY SCHEMALESS;",
)

major = int((await demo.version()).rsplit("-", 1)[-1].split(".")[0])  # "surrealdb-3.3.4" -> 3
print("Connections:", SurrealDBConnectionManager.list_connections())
print(f"Server: SurrealDB {major}.x")

Connections: ['default', 'views_demo']
Server: SurrealDB 3.x


## 3. Models

`Order` is an ordinary model: the source table. The three others are views of it, each declared with `view_query`:

- `OrderStats` groups the orders by status, with a count and a sum (`GROUP BY`);
- `PaidOrder` keeps a filtered copy of the paid orders (`WHERE`, no grouping);
- `OrderTotals` sums every order into one record (`GROUP ALL`).

A grouped view's record id is the array of its group values (`order_stats:['paid']`, and `order_totals:[]` for
`GROUP ALL`), so the views declare `id: RecordID | None = None`. Sums are declared `float`: SurrealDB 2.x computes an
`int` sum of integers, which a `float` field reads as a float. The query is checked when the class is defined: one
`SELECT` statement, not a `SELECT VALUE`, and in a grouped view no function SurrealDB cannot keep up to date, such as
`math::median()`; a mistake raises at the `class` statement. It is written into the DDL as it is: trusted application
code, never user input.

In [2]:
from surrealdb import RecordID

from surreal_orm_lite import BaseSurrealModel, SurrealConfigDict


class Order(BaseSurrealModel):
    model_config = SurrealConfigDict(table_name="orders", connection="views_demo")

    id: str | None = None
    status: str
    amount: float
    customer: str | None = None


class OrderStats(BaseSurrealModel):
    model_config = SurrealConfigDict(
        table_name="order_stats",
        connection="views_demo",
        view_query="SELECT status, count() AS total, math::sum(amount) AS revenue FROM orders GROUP BY status",
    )

    id: RecordID | None = None
    status: str
    total: int
    revenue: float


class PaidOrder(BaseSurrealModel):
    model_config = SurrealConfigDict(
        table_name="paid_orders",
        connection="views_demo",
        view_query="SELECT customer, amount FROM orders WHERE status = 'paid'",
    )

    id: RecordID | None = None
    customer: str | None = None
    amount: float


class OrderTotals(BaseSurrealModel):
    model_config = SurrealConfigDict(
        table_name="order_totals",
        connection="views_demo",
        view_query="SELECT count() AS orders, math::sum(amount) AS revenue FROM orders GROUP ALL",
    )

    id: RecordID | None = None
    orders: int
    revenue: float


print("Order is a view:", Order.get_view_query() is not None)
print("OrderStats reads:", OrderStats.get_view_query())

Order is a view: False
OrderStats reads: SELECT status, count() AS total, math::sum(amount) AS revenue FROM orders GROUP BY status


## 4. Apply the views

`view_ddl()` renders the statement without a connection: the table name quoted, the query inside `AS (…)`. Three
orders are saved, then `define_view()` applies each view and returns the statement it applied: SurrealDB fills the
view from the orders already there. It writes `DEFINE TABLE OVERWRITE`, so it is idempotent — run it at start-up or
deployment; each call recomputes the view (`overwrite=False` writes `IF NOT EXISTS` and keeps a view already defined).

In [3]:
for view in (OrderStats, PaidOrder, OrderTotals):
    print(view.view_ddl())

for key, status, amount, customer in [("o1", "paid", 10.0, "ada"), ("o2", "paid", 5.5, "bob"), ("o3", "new", 1.0, "ada")]:
    await Order(id=key, status=status, amount=amount, customer=customer).save()

first = await OrderStats.define_view()
second = await OrderStats.define_view()  # idempotent: OVERWRITE recomputes the view
await PaidOrder.define_view()
await OrderTotals.define_view()
print()
print("applied twice, the same statement:", first == second == OrderStats.view_ddl())

DEFINE TABLE OVERWRITE `order_stats` AS (SELECT status, count() AS total, math::sum(amount) AS revenue FROM orders GROUP BY status);
DEFINE TABLE OVERWRITE `paid_orders` AS (SELECT customer, amount FROM orders WHERE status = 'paid');
DEFINE TABLE OVERWRITE `order_totals` AS (SELECT count() AS orders, math::sum(amount) AS revenue FROM orders GROUP ALL);

applied twice, the same statement: True


## 5. Read the views

A view reads like any model. `PaidOrder` keeps the source records' ids (`paid_orders:o1`); `OrderTotals`, grouped with
`GROUP ALL`, holds one record, read with `first()`.

In [4]:
async def show_stats() -> None:
    """Print OrderStats, one line per status."""
    for stats in await OrderStats.objects().order_by("status").exec():
        print(f"  {stats.status}: {stats.total} order(s), revenue {stats.revenue}")


print("stats:")
await show_stats()
print("paid:", [(str(order.id), order.customer, order.amount) for order in await PaidOrder.objects().order_by("amount").exec()])
totals = await OrderTotals.objects().first()
print("totals:", totals.orders, "orders, revenue", totals.revenue, "| id:", totals.id)

stats:
  new: 1 order(s), revenue 1.0
  paid: 2 order(s), revenue 15.5
paid: [('paid_orders:o2', 'bob', 5.5), ('paid_orders:o1', 'ada', 10.0)]
totals: 3 orders, revenue 16.5 | id: order_totals:[]


SurrealDB keeps the views up to date in the statement that writes an order: a new status creates a group, an amount
change updates its group, and deleting the last order of a group removes that group. `get(["paid"])` looks a group up
by its id, the array of its group values; `filter(id=["paid"])` does the same.

In [5]:
await Order(id="o4", status="shipped", amount=7.0, customer="bob").save()  # a new group
await (await Order.objects().get("o2")).merge(amount=6.5)  # the paid group's revenue changes
await (await Order.objects().get("o3")).delete()  # the last 'new' order: its group is gone

print("stats:")
await show_stats()
paid = await OrderStats.objects().get(["paid"])
print("get(['paid']):", paid.id, "|", type(paid.id).__name__, "| total:", paid.total)

stats:
  paid: 2 order(s), revenue 16.5
  shipped: 1 order(s), revenue 7.0
get(['paid']): order_stats:['paid'] | RecordID | total: 2


## 6. Writes are refused

SurrealDB computes a view's records, so every ORM write of a view model raises `TypeError` before any request — the
instance methods (`save()`, `merge()`, `delete()`, the atomic helpers…) and the QuerySet's (`bulk_update()`,
`bulk_delete()`, `get_or_create()`…). The message starts with the full SurrealDB-ORM's `Cannot modify materialized
view '<table>'` and names the call. Write to the source table instead. Raw SurrealQL is not checked (SurrealDB 3.2 and
later refuse a raw write to a view; 2.x and 3.1 accept it and change the view's records).

In [6]:
try:
    await OrderStats(status="paid", total=0, revenue=0.0).save()
except TypeError as error:
    print(error)

try:
    await paid.merge(total=0)
except TypeError as error:
    print(error)

try:
    await OrderStats.objects().filter(status="paid").bulk_update(total=0)
except TypeError as error:
    print(error)

print()
print("stats unchanged:")
await show_stats()

Cannot modify materialized view 'order_stats': OrderStats.save() writes to it, and SurrealDB computes a view's records from its view_query. Write to the tables it selects from instead.
Cannot modify materialized view 'order_stats': OrderStats.merge() writes to it, and SurrealDB computes a view's records from its view_query. Write to the tables it selects from instead.
Cannot modify materialized view 'order_stats': OrderStats.objects().bulk_update() writes to it, and SurrealDB computes a view's records from its view_query. Write to the tables it selects from instead.

stats unchanged:
  paid: 2 order(s), revenue 16.5
  shipped: 1 order(s), revenue 7.0


## 7. A live dashboard

Over WebSocket, `watch()` on a view model receives the records SurrealDB recomputes: here a new `refunded` group, a
change of the `paid` group, then the `refunded` group emptied by a delete. The cell reads the notifications up to the
`DELETE` (at most 5 s). `watch()` yields the server's envelopes as received: the action, the view record's id
(`record`) and its values (`result`).

In [7]:
async def until_delete(stream, timeout: float = 5.0) -> list[dict]:
    """The envelopes of *stream* up to the first DELETE, read within *timeout* seconds."""

    async def read() -> list[dict]:
        seen = []
        async for envelope in stream:
            seen.append(envelope)
            if envelope["action"] == "DELETE":
                break
        return seen

    return await asyncio.wait_for(read(), timeout)


async with OrderStats.objects().watch() as stream:
    await Order(id="o5", status="refunded", amount=2.0, customer="ada").save()  # a new group
    await Order(id="o6", status="paid", amount=4.0, customer="eve").save()  # the paid group changes
    await (await Order.objects().get("o5")).delete()  # the refunded group is emptied
    envelopes = await until_delete(stream)

for envelope in envelopes:
    result = envelope["result"]
    print(f"{envelope['action']:<6} {envelope['record']}: total {result['total']}, revenue {result['revenue']}")

CREATE order_stats:['refunded']: total 1, revenue 2.0
UPDATE order_stats:['paid']: total 3, revenue 20.5
DELETE order_stats:['refunded']: total 1, revenue 2.0


The emptied group is where the lines differ: SurrealDB 3.x notifies one `DELETE` carrying the group's last values;
SurrealDB 2.x notifies an `UPDATE` with the emptied values (a count of 0) and then the `DELETE`. Events defined on a
view see the same sequence. On 3.x the envelope's `result` has no `id` (read `record`), and a live query on a view
filtered on `id` never notifies: filter on the view's fields, e.g. `filter(status="paid")`. `live()` yields typed
`OrderStats` instances carrying the record id on both lines.

## 8. Transactions

A source write inside `transaction()` updates the view in the same transaction. Over WebSocket on SurrealDB 3.x the
transaction is **interactive**: the block reads the view with its own write (`objects(tx=tx)`), which nobody outside
the block sees before the commit. Elsewhere (SurrealDB 2.x, HTTP, the embedded engine) the transaction is
**buffered**: the writes are sent at the end of the block, reads inside it are unsupported, and the view is updated at
the commit. The cell probes which kind the server offers and branches; it runs on both lines.

In [8]:
async def native_tx_supported(client) -> bool:
    """True if the server exposes the SDK's native transaction RPC (SurrealDB 3.x)."""
    try:
        txn = await client.begin()
    except Exception:
        return False
    with contextlib.suppress(Exception):
        await client.cancel(txn)
    return True


interactive = await native_tx_supported(demo)
print("Interactive transactions available (SurrealDB 3.x):", interactive)

async with SurrealDBConnectionManager.transaction("views_demo") as tx:
    await Order(id="o7", status="shipped", amount=3.0, customer="eve").save(tx=tx)
    if interactive:
        inside = await OrderStats.objects(tx=tx).get(["shipped"])
        outside = await OrderStats.objects().get(["shipped"])
        print("shipped, read in the block:", inside.total, "| read outside, before the commit:", outside.total)
    else:
        print("Buffered transaction: the write is sent at the end of the block; reads inside it are unsupported.")

print("shipped, after the commit:", (await OrderStats.objects().get(["shipped"])).total)

Interactive transactions available (SurrealDB 3.x): True
shipped, read in the block: 2 | read outside, before the commit: 1
shipped, after the commit: 2


## 9. What depends on the server line

SurrealDB 3.x keeps an expression around an aggregate up to date in a grouped view, such as an average written
`math::sum(amount) / count()`. SurrealDB 2.x (and the embedded engine) keep only whole aggregates and `GROUP BY`
fields up to date and compute anything else once per source record: the view would hold a wrong value, or every
write to the source table would be refused. So on those engines `define_view()` refuses such a view before any DDL,
with `SurrealDbError`; the same holds for `math::stddev()`, a field outside `GROUP BY` and outside an aggregate (3.x
refuses that one too) and a `GROUP BY` on an alias. Below, the average view is defined and read on 3.x; on 2.x the
cell prints the refusal.

In [9]:
from surreal_orm_lite import SurrealDbError


class OrderAverage(BaseSurrealModel):
    model_config = SurrealConfigDict(
        table_name="order_averages",
        connection="views_demo",
        view_query="SELECT status, math::sum(amount) / count() AS average FROM orders GROUP BY status",
    )

    id: RecordID | None = None
    status: str
    average: float


if major >= 3:
    await OrderAverage.define_view()
    for row in await OrderAverage.objects().order_by("status").exec():
        print(f"{row.status}: average {round(row.average, 2)}")
else:
    try:
        await OrderAverage.define_view()
    except SurrealDbError as error:  # refused before any DDL: nothing was defined
        print(error)

paid: average 6.83
shipped: average 5.0


The portable way, on every line: select the whole aggregates — `math::sum(amount) AS revenue` and `count() AS total`,
as `OrderStats` does — and divide when reading the model.

In [10]:
for stats in await OrderStats.objects().order_by("status").exec():
    print(f"{stats.status}: average {round(stats.revenue / stats.total, 2)}")

paid: average 6.83
shipped: average 5.0


## 10. The table guard

`define_view()` never turns a table that holds records into a view: SurrealDB would delete them. Below, a view model
is given the source table's name by mistake. Before any DDL, `define_view()` reads the database's definitions, sees
that `orders` is an ordinary table holding records, and raises `SurrealDbError` — with the `raw_query("REMOVE TABLE
…")` to run if the records are not needed. The orders are untouched. (An ordinary table **without** records, such as
one `define_events()` creates, is turned into the view; `remove_view()` refuses a table that is not a view.)

In [11]:
class OrdersByMistake(BaseSurrealModel):
    model_config = SurrealConfigDict(
        table_name="orders",  # the source table itself
        connection="views_demo",
        view_query="SELECT customer, amount FROM orders WHERE amount > 5",
    )

    id: RecordID | None = None
    customer: str | None = None
    amount: float


try:
    await OrdersByMistake.define_view()
except SurrealDbError as error:  # refused before any DDL
    print(error)

print()
print("orders still there:", await Order.objects().count())

OrdersByMistake.define_view(): the table 'orders' exists, is not a view and holds records; defining the view would delete them, with the table's fields, indexes and events. If they are not needed, remove the table first (await OrdersByMistake.raw_query("REMOVE TABLE `orders`;")); otherwise give the model another table_name. Nothing was defined.

orders still there: 5


## 11. The query cache

A cached read of a view model (`cache(ttl)`, v0.28.0) is dropped by **every** ORM write: a write to a source table
changes the view, and the ORM does not know which tables a view reads. Below, the second read is answered from the
cache; after an order is saved, the same read goes to the database again and shows the new total.

In [12]:
from surreal_orm_lite import QueryCache


def paid_total(rows: list[OrderStats]) -> int:
    """The paid group's count in *rows*."""
    return next(row.total for row in rows if row.status == "paid")


hits = QueryCache.stats()["hits"]
cached = await OrderStats.objects().order_by("status").cache(60).exec()
again = await OrderStats.objects().order_by("status").cache(60).exec()  # answered from the cache
print("paid:", paid_total(cached), "| from the cache:", QueryCache.stats()["hits"] - hits, "read(s)")

await Order(id="o8", status="paid", amount=1.5, customer="bob").save()  # a write to the source table
fresh = await OrderStats.objects().order_by("status").cache(60).exec()  # the cached read was dropped
print("paid after the write:", paid_total(fresh), "| from the cache:", QueryCache.stats()["hits"] - hits, "read(s)")

paid: 3 | from the cache: 1 read(s)
paid after the write: 4 | from the cache: 1 read(s)


## 12. A view over a view

A view may read another view: `StatsTotals` copies the count of each `OrderStats` group, and SurrealDB updates it when
`OrderStats` changes. Removing the inner view while the outer one reads it is refused on both lines, in two ways:
SurrealDB 3.x refuses to remove a table that a view reads, and `remove_view()` passes that refusal on
(`Cannot delete table … on which a view is defined`); SurrealDB 2.x would remove it and turn `StatsTotals` into an
ordinary table that keeps its last records and no longer updates, so `remove_view()` raises before any DDL, naming the
views that read it. Either way it is a `SurrealDbError`, and nothing is removed.

In [13]:
class StatsTotals(BaseSurrealModel):
    model_config = SurrealConfigDict(
        table_name="stats_totals", connection="views_demo", view_query="SELECT total FROM order_stats"
    )

    id: RecordID | None = None
    total: int


await StatsTotals.define_view()
print("stats_totals:", [(str(row.id), row.total) for row in await StatsTotals.objects().order_by("total").exec()])

try:
    await OrderStats.remove_view()
except SurrealDbError as error:
    print(error)
print("order_stats still read:", await OrderStats.objects().count(), "groups")

stats_totals: [("stats_totals:['shipped']", 2), ("stats_totals:['paid']", 4)]
Can't apply view removal: 'order_stats' -> Invalid query: Cannot delete table `order_stats` on which a view is defined, table(s) `stats_totals` are defined as a view on this table.
order_stats still read: 2 groups


**Remove the views that read another view first.** Once `StatsTotals` is removed, `OrderStats` can be removed too (the
cleanup does it).

In [14]:
print(await StatsTotals.remove_view())

REMOVE TABLE IF EXISTS `stats_totals`;


## 13. Read the views back

v0.30.0's `DatabaseIntrospector` reads a view's query back (`TableState.view_query`), and `generate_models_from_db()`
generates a view model from it: `view_query=…` in its `model_config`, `id: RecordID | None` and the
`# Read-only materialized view` comment. A view's table defines no field, so the generated class declares only `id`
and keeps the view's columns as extra fields (`extra="allow"`): add typed fields when you review it. The file is
written to a temporary directory, whose path is not printed.

In [15]:
import pathlib
import tempfile

from surreal_orm_lite import DatabaseIntrospector, generate_models_from_db

schema = await DatabaseIntrospector("views_demo", tables=["order_stats"]).introspect()
print("view_query:", schema.tables["order_stats"].view_query)
print("issues:", schema.issues)
print()

with tempfile.TemporaryDirectory() as directory:
    path = pathlib.Path(directory) / "views_models.py"
    source = await generate_models_from_db(path, "views_demo", tables=["order_stats"])
    print(source)

view_query: SELECT status, count() AS total, math::sum(amount) AS revenue FROM orders GROUP BY status
issues: ()



"""Models generated by surreal_orm_lite.generate_models_from_db() from a SurrealDB schema.

Review them before use. They describe tables that already exist: define_table()
keeps the clauses of the database they do not declare (a COMPUTED field stays
COMPUTED, a server DEFAULT stays), so it can be called on them; do not call
define_computed_fields() on them, it would replace the database's field definitions.
"""

from surreal_orm_lite import BaseSurrealModel, SurrealConfigDict
from surrealdb import RecordID


class OrderStats(BaseSurrealModel):
    # Read-only materialized view
    # SCHEMALESS: fields not declared here are kept (extra="allow")
    model_config = SurrealConfigDict(
        table_name="order_stats",
        view_query="SELECT status, count() AS total, math::sum(amount) AS revenue FROM orders GROUP BY status",
        connection="views_demo",
        extra="allow",
    )

    id: RecordID | None = None



## 14. Cleanup

Remove the views — any view that reads another view first; `remove_view()` writes `REMOVE TABLE IF EXISTS`, so a view
never defined (the average one on 2.x) is no error. Then remove the `examples_views` database, unregister
`views_demo` — `remove_connection()` closes its client — and close every connection with `close_all_connections()`.

In [16]:
for view in (StatsTotals, OrderAverage, PaidOrder, OrderTotals, OrderStats):  # a view over another view first
    print(await view.remove_view())

await apply_ddl(demo, "REMOVE DATABASE IF EXISTS examples_views;")
await SurrealDBConnectionManager.remove_connection("views_demo")
print("Connections:", SurrealDBConnectionManager.list_connections())
await SurrealDBConnectionManager.close_all_connections()
print("Done")

REMOVE TABLE IF EXISTS `stats_totals`;
REMOVE TABLE IF EXISTS `order_averages`;
REMOVE TABLE IF EXISTS `paid_orders`;
REMOVE TABLE IF EXISTS `order_totals`;


REMOVE TABLE IF EXISTS `order_stats`;
Connections: ['default']
Done
